# ruMMLU (ai-forever/MERA) — Data Inspection

Checks format, subject distribution, language quality, and split boundaries.
Use only the `train` split for SFT — the test split is closed and must not be used for training.

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset('ai-forever/MERA', 'rummlu')
print(ds)
print('Splits:', list(ds.keys()))

In [ ]:
# Inspect raw structure
ex = ds['train'][0]
print('Keys:', list(ex.keys()))
print()
for k, v in ex.items():
    print(f'{k}: {repr(v)[:200]}')

In [ ]:
# Parse inputs dict and build a flat dataframe
rows = []
for split in ds.keys():
    for ex in ds[split]:
        inp = ex['inputs']
        rows.append({
            'split':    split,
            'subject':  inp.get('subject', ''),
            'text':     inp.get('text', ''),
            'option_a': inp.get('option_a', ''),
            'option_b': inp.get('option_b', ''),
            'option_c': inp.get('option_c', ''),
            'option_d': inp.get('option_d', ''),
            'answer':   str(ex.get('outputs', '')).strip().upper(),
            'meta':     ex.get('meta', {}),
        })

df = pd.DataFrame(rows)
print(df.shape)
df.head(3)

In [ ]:
# Split sizes
print(df.groupby('split').size())

In [ ]:
# Subject distribution in train split
train = df[df['split'] == 'train']
print(f'Train size: {len(train)}')
print()
print(train['subject'].value_counts().to_string())

In [ ]:
# Answer distribution — should be roughly uniform A/B/C/D
print(train['answer'].value_counts())

In [ ]:
# Check for malformed answers (not A/B/C/D)
bad_answer = train[~train['answer'].isin(['A', 'B', 'C', 'D'])]
print(f'Bad answers: {len(bad_answer)}')
if len(bad_answer):
    display(bad_answer[['subject', 'text', 'answer']].head(10))

In [ ]:
# Check for empty fields
for col in ['text', 'option_a', 'option_b', 'option_c', 'option_d']:
    n_empty = (train[col].str.strip() == '').sum()
    print(f'Empty {col}: {n_empty}')

In [ ]:
# Check for encoding corruption
import re
corruption = re.compile(r'[\ufffd\x00-\x08\x0b\x0c\x0e-\x1f]')
for col in ['text', 'option_a', 'option_b', 'option_c', 'option_d']:
    n = train[col].apply(lambda x: bool(corruption.search(str(x)))).sum()
    print(f'Corruption in {col}: {n}')

In [ ]:
# Spot-check a few examples per subject area
for subject in train['subject'].value_counts().head(5).index:
    row = train[train['subject'] == subject].iloc[0]
    print(f'=== {subject} ===')
    print(f'Q: {row["text"]}')
    for letter, col in zip('ABCD', ['option_a', 'option_b', 'option_c', 'option_d']):
        print(f'  {letter}) {row[col]}')
    print(f'Answer: {row["answer"]}')
    print()

In [ ]:
# Check question length distribution
train['q_len'] = train['text'].str.len()
print(train['q_len'].describe())

# Flag very short questions (possible truncation)
short = train[train['q_len'] < 20]
print(f'Very short questions (<20 chars): {len(short)}')
if len(short):
    display(short[['subject', 'text', 'answer']].head(10))

In [ ]:
# Summary
n_clean = len(train[
    train['answer'].isin(['A', 'B', 'C', 'D']) &
    (train['text'].str.strip() != '') &
    (train['option_a'].str.strip() != '')
])
print(f'Total train rows:  {len(train)}')
print(f'Clean rows:        {n_clean}')
print(f'Subjects:          {train["subject"].nunique()}')
print()
print('Verdict: safe to write loader directly from train split.')
print('Do NOT use test split for training (closed benchmark).')